# Fase 10 — Síntesis de evidencia y transferibilidad a anchoveta

Esta fase integra los 97 hallazgos validados con la calidad de sus fuentes y evalúa su uso potencial para el stock norte-centro de anchoveta peruana.

Principios:

- transferibilidad no equivale a calidad, relevancia, implementación ni efectividad;
- una recomendación o propuesta no prueba implementación;
- una actividad o producto no prueba resultado o impacto;
- una proyección no se codifica como tendencia observada;
- la fortaleza de evidencia se hereda de la evaluación de calidad validada;
- las recomendaciones finales requieren revisión experta y no se generan automáticamente a partir de conteos.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from evidence_review.transferability_synthesis import (
    build_evidence_gap_matrix,
    build_evidence_synthesis_matrix,
    build_management_option_summary,
    build_recommendation_candidates,
    build_synthesis_theme_summary,
    export_transferability_prompts_jsonl,
    initialise_transferability_sheet,
    load_transferability_config,
    merge_transferability_assessments,
    read_csv_robust,
    read_transferability_responses_jsonl,
    split_transferability_outputs,
    transferability_summary,
    validate_transferability_sheet,
)

CONFIG_PATH = ROOT / "config" / "transferability_synthesis.yml"
config = load_transferability_config(
    CONFIG_PATH,
    project_root=ROOT,
)
paths = config["paths"]

INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Configuration: {CONFIG_PATH.relative_to(ROOT)}")


## 1. Cargar los hallazgos con calidad validada

La entrada principal es `finding_quality_link.csv`. Debe contener los 97 hallazgos de la fase 08 y las variables de calidad de la fase 09 con prefijo `quality_`.


In [ ]:
findings_path = ROOT / paths["findings_with_quality_csv"]
working_path = ROOT / paths["transferability_working_csv"]

findings, findings_encoding = read_csv_robust(findings_path)

existing = pd.DataFrame()
if working_path.exists():
    existing, working_encoding = read_csv_robust(working_path)
else:
    working_encoding = "not_loaded"

missing_quality = (
    findings.get(
        "quality_overall_rating",
        pd.Series("", index=findings.index),
    )
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(f"Findings with quality: {len(findings)} [{findings_encoding}]")
print(f"Sources represented: {findings['source_id'].nunique()}")
print(f"Missing validated quality: {missing_quality}")
print(f"Existing transferability rows: {len(existing)} [{working_encoding}]")


## 2. Inicializar la evaluación por hallazgo

Se crea una fila por `finding_id`. Las filas humanas `accepted`, `corrected` o `rejected` se preservan. La fortaleza de evidencia se deriva de `quality_overall_rating`.


In [ ]:
sheet = initialise_transferability_sheet(
    findings,
    config,
    existing,
)

if not working_path.exists():
    sheet.to_csv(
        working_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"Created: {working_path.relative_to(ROOT)}")
else:
    print(
        "Existing transferability_assessment_working.csv preserved; "
        "initialization did not overwrite it."
    )

display(transferability_summary(sheet))
display(
    sheet[
        [
            "finding_id",
            "source_id",
            "finding_type",
            "quality_overall_rating",
            "score__evidence_strength",
            "transferability_status",
            "human_validation_status",
        ]
    ].head(20)
)


## 3. Generar prompts auditables

Se genera un prompt por hallazgo. Esta sección no llama a ningún modelo externo. Cada prompt conserva el hallazgo, su extracto, la calidad validada y el perfil de la pesquería objetivo.


In [ ]:
prompts_path = ROOT / paths["prompts_jsonl"]
export_transferability_prompts_jsonl(
    sheet,
    config,
    prompts_path,
)

print(f"Prompts generated: {len(sheet)}")
print(f"Saved: {prompts_path.relative_to(ROOT)}")


## 4. Importar evaluaciones asistidas opcionales

El archivo esperado es `transferability_assessment_model_responses.jsonl`, con una evaluación por hallazgo. La importación está desactivada por defecto.


In [ ]:
IMPORT_MODEL_RESPONSES = False
OVERWRITE_HUMAN_VALIDATED = False

responses_path = ROOT / paths["model_responses_jsonl"]

if IMPORT_MODEL_RESPONSES:
    if not responses_path.exists():
        raise FileNotFoundError(responses_path)

    incoming = read_transferability_responses_jsonl(
        responses_path,
        sheet,
        config,
    )
    sheet = merge_transferability_assessments(
        sheet,
        incoming,
        config,
        overwrite_human_validated=OVERWRITE_HUMAN_VALIDATED,
    )
    sheet.to_csv(
        working_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"Imported finding assessments: {len(incoming)}")
    print(f"Updated: {working_path.relative_to(ROOT)}")
else:
    print(
        "Model-response import disabled. Set IMPORT_MODEL_RESPONSES = True "
        "only after reviewing the JSONL."
    )


## 5. Recargar y validar

La hoja se recarga desde disco inmediatamente antes de validar para evitar exportar una versión antigua mantenida en memoria.


In [ ]:
sheet, reloaded_encoding = read_csv_robust(working_path)

print(
    f"Reloaded assessments from: "
    f"{working_path.relative_to(ROOT)} [{reloaded_encoding}]"
)
print(
    sheet["human_validation_status"]
    .value_counts(dropna=False)
)

issues = validate_transferability_sheet(
    sheet,
    config,
)

issues_path = ROOT / paths["issues_csv"]
issues.to_csv(
    issues_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Validation issues: {len(issues)}")
display(issues.head(100))
display(transferability_summary(sheet))


## 6. Exportar la síntesis

Las matrices se construyen únicamente con evaluaciones `completed` y validación humana `accepted` o `corrected`. Los conteos describen cobertura de evidencia, no tamaños de efecto.


In [ ]:
sheet, _ = read_csv_robust(working_path)

groups = split_transferability_outputs(sheet)
matrix = build_evidence_synthesis_matrix(groups["validated"])
theme_summary = build_synthesis_theme_summary(matrix)
management_summary = build_management_option_summary(matrix)
gap_matrix = build_evidence_gap_matrix(matrix)
recommendation_candidates = build_recommendation_candidates(matrix)
flow = transferability_summary(sheet)

output_frames = {
    paths["transferability_csv"]: groups["transferability"],
    paths["validated_transferability_csv"]: groups["validated"],
    paths["pending_validation_csv"]: groups["pending_validation"],
    paths["rejected_transferability_csv"]: groups["rejected"],
    paths["synthesis_matrix_csv"]: matrix,
    paths["synthesis_theme_summary_csv"]: theme_summary,
    paths["management_option_summary_csv"]: management_summary,
    paths["evidence_gap_matrix_csv"]: gap_matrix,
    paths["recommendation_candidates_csv"]: recommendation_candidates,
    paths["transferability_flow_csv"]: flow,
}

for relative_path, frame in output_frames.items():
    path = ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"{path.relative_to(ROOT)}: {len(frame)}")

display(flow)
display(theme_summary)
display(management_summary.head(30))


## Criterio para avanzar a la fase 11

La validación final y generación de productos puede comenzar cuando:

1. `Validation issues = 0`;
2. los 97 hallazgos tienen evaluación `completed` o una justificación documentada;
3. las evaluaciones utilizadas están `accepted` o `corrected`;
4. la fortaleza de evidencia coincide con la calidad validada de la fuente;
5. las categorías y puntuaciones son consistentes;
6. propuesta, implementación y efectividad permanecen diferenciadas;
7. los hallazgos de transferibilidad alta han sido revisados;
8. las recomendaciones candidatas se interpretan como insumos para revisión experta.

La siguiente fase será:

```text
notebooks/11_validate_and_build_products.ipynb
```
